In [1]:
import gymnasium as gym 
import numpy as np 
import math 
from torch.utils.tensorboard import SummaryWriter 
from gymnasium.wrappers import RecordVideo

In [2]:
# Create training environment
env = gym.make("LunarLander-v3")
writer = SummaryWriter(log_dir="runs/lunarlander_mc")

# Discretization bins for 8-dimensional state
NUM_BINS = (8, 8, 8, 8, 8, 8, 2, 2)
TOTAL_STATES = np.prod(NUM_BINS)
print(TOTAL_STATES)

1048576


In [3]:
# Pre-defined observation bounds
obs_space_low = np.array([-1.5, -1.5, -1.5, -2.0, -math.pi, -5.0, 0.0, 0.0])
obs_space_high = np.array([1.5, 1.5, 1.5, 2.0, math.pi, 5.0, 1.0, 1.0])
obs_range = obs_space_high - obs_space_low

print(obs_space_high.shape)
print(obs_range)

(8,)
[ 3.          3.          3.          4.          6.28318531 10.
  1.          1.        ]


In [4]:
def discretize(obs):
    ratios = np.clip((obs - obs_space_low) / obs_range, 0, 0.999)
    print(ratios, flush=True)
    bins = (ratios * NUM_BINS).astype(int)
    
    # Convert to single index
    multipliers = np.array([NUM_BINS[1] * NUM_BINS[2] * NUM_BINS[3] * NUM_BINS[4] * NUM_BINS[5] * NUM_BINS[6] * NUM_BINS[7],
                           NUM_BINS[2] * NUM_BINS[3] * NUM_BINS[4] * NUM_BINS[5] * NUM_BINS[6] * NUM_BINS[7],
                           NUM_BINS[3] * NUM_BINS[4] * NUM_BINS[5] * NUM_BINS[6] * NUM_BINS[7],
                           NUM_BINS[4] * NUM_BINS[5] * NUM_BINS[6] * NUM_BINS[7],
                           NUM_BINS[5] * NUM_BINS[6] * NUM_BINS[7],
                           NUM_BINS[6] * NUM_BINS[7],
                           NUM_BINS[7],
                           1])
    print(multipliers)
    print(bins, flush=True)
    return np.dot(bins, multipliers)

In [5]:
state, _ = env.reset()
state = discretize(state)

[0.50049728 0.97041531 0.55036757 0.50361963 0.49972593 0.4965773
 0.         0.        ]
[131072  16384   2048    256     32      4      2      1]
[4 7 4 4 3 3 0 0]


In [6]:
state

np.int64(648300)

In [7]:
# Initialize Q-table and visit counts for Monte Carlo
Q = np.zeros((TOTAL_STATES, env.action_space.n))
returns_sum = np.zeros((TOTAL_STATES, env.action_space.n))
returns_count = np.zeros((TOTAL_STATES, env.action_space.n))

# Hyperparameters
GAMMA = 0.99
EPSILON = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.9995
NUM_EPISODES = 20

# Reward tracking
all_rewards = []


In [8]:
print(Q.shape)

(1048576, 4)


In [9]:
# Epsilon-greedy policy
def choose_action(state, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(env.action_space.n)
    return np.argmax(Q[state])

print("Starting LunarLander Monte Carlo training...")
print(f"State space size: {TOTAL_STATES:,}")
print(f"Total episodes to train: {NUM_EPISODES:,}")
print("=" * 50)

# ---------------------------
# Monte Carlo Training
# ---------------------------
for episode in range(NUM_EPISODES):
    # Generate episode
    episode_states = []
    episode_actions = []
    episode_rewards = []
    
    state, _ = env.reset()
    state = discretize(state)
    done = False
    steps = 0

    # Collect episode data
    while not done and steps < 10:
        action = choose_action(state, EPSILON)
        next_state, reward, done, truncated, _ = env.step(action)
        
        episode_states.append(state)
        episode_actions.append(action)
        episode_rewards.append(reward)
        
        state = discretize(next_state)
        steps += 1

    # Calculate returns for this episode
    episode_length = len(episode_rewards)
    total_reward = sum(episode_rewards)
    all_rewards.append(total_reward)
    
    # Calculate discounted returns (working backwards)
    G = 0
    for t in range(episode_length - 1, -1, -1):
        G = GAMMA * G + episode_rewards[t]
        
        state_t = episode_states[t]
        action_t = episode_actions[t]
        
        
        returns_sum[state_t, action_t] += G
        returns_count[state_t, action_t] += 1
            
        # Update Q-value with average return
        Q[state_t, action_t] = returns_sum[state_t, action_t] / returns_count[state_t, action_t]

    # Epsilon decay
    EPSILON = max(EPSILON * EPSILON_DECAY, EPSILON_MIN)

    # Logging
    if episode % 50 == 0:
        writer.add_scalar("Train/EpisodeReward", total_reward, episode)
        if episode >= 100:
            avg_reward = np.mean(all_rewards[-5:])
            writer.add_scalar("Train/MovingAverageReward", avg_reward, episode)
            writer.add_scalar("Train/Epsilon", EPSILON, episode)

    # Simple progress tracking
    if (episode + 1) % 1000 == 0:
        recent_avg = np.mean(all_rewards[-5:]) if len(all_rewards) >= 5 else np.mean(all_rewards)
        states_visited = np.sum(returns_count > 0)
        print(f"Episode {episode + 1:5d}/{NUM_EPISODES} - Avg Reward: {recent_avg:.1f} - States Visited: {states_visited}")

env.close()

print("Training completed! Starting evaluation...")

Starting LunarLander Monte Carlo training...
State space size: 1,048,576
Total episodes to train: 20
[0.50090984 0.97373835 0.59215039 0.61438897 0.49949771 0.49373797
 0.         0.        ]
[131072  16384   2048    256     32      4      2      1]
[4 7 4 4 3 3 0 0]
[0.50181967 0.97697719 0.59202586 0.60795986 0.49900525 0.49381111
 0.         0.        ]
[131072  16384   2048    256     32      4      2      1]
[4 7 4 4 3 3 0 0]
[0.50269912 0.98001834 0.58821118 0.60137083 0.49887873 0.49840993
 0.         0.        ]
[131072  16384   2048    256     32      4      2      1]
[4 7 4 4 3 3 0 0]
[0.5035525  0.98285774 0.58494514 0.59464697 0.49906472 0.50233748
 0.         0.        ]
[131072  16384   2048    256     32      4      2      1]
[4 7 4 4 3 4 0 0]
[0.50440588 0.98549708 0.58494383 0.58797986 0.49925063 0.50233638
 0.         0.        ]
[131072  16384   2048    256     32      4      2      1]
[4 7 4 4 3 4 0 0]
[0.50529076 0.98793677 0.58889822 0.58132189 0.49905789 0.497577